# Defects Data — Exploratory Data Analysis

This notebook performs EDA on `defects_data.csv`: data overview, missing values, distributions, correlations, and basic outlier checks.

In [1]:
# Imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

data_path = 'defects_data.csv'
df = pd.read_csv(data_path, parse_dates=['defect_date'], infer_datetime_format=True, dayfirst=False)
df.head()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Basic info and summary
print('Shape:', df.shape)
print('
Info:')
df.info()

print('
Describe (numeric):')
display(df.describe())

print('
Describe (all):')
display(df.describe(include='all'))

In [ ]:
# Missing values and duplicates
print('Missing values per column:')
print(df.isnull().sum())

print('
Duplicate rows:')
print(df.duplicated().sum())

In [ ]:
# Parse date and create date features
df['defect_date'] = pd.to_datetime(df['defect_date'], errors='coerce')
df['year'] = df['defect_date'].dt.year
df['month'] = df['defect_date'].dt.month
df['weekday'] = df['defect_date'].dt.day_name()
df[['defect_date','year','month','weekday']].head()

In [ ]:
# Categorical distributions
cat_cols = ['defect_type','defect_location','severity','inspection_method']
for col in cat_cols:
    print('
==', col, '==')
    print(df[col].value_counts())
    plt.figure(figsize=(6,3))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Numeric distributions and relationship with severity
plt.figure(figsize=(8,4))
sns.histplot(df['repair_cost'], kde=True)
plt.title('Repair cost distribution')
plt.show()

print(df['repair_cost'].describe())

plt.figure(figsize=(8,4))
sns.boxplot(x='severity', y='repair_cost', data=df, order=['Minor','Moderate','Critical'])
plt.title('Repair cost by severity')
plt.show()

In [ ]:
# Correlations (numeric features)
numeric = df.select_dtypes(include=[np.number])
plt.figure(figsize=(6,4))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Numeric Correlations')
plt.show()

In [ ]:
# Outlier check for repair_cost using IQR
q1 = df['repair_cost'].quantile(0.25)
q3 = df['repair_cost'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr
outliers = df[(df['repair_cost'] < lower) | (df['repair_cost'] > upper)]
print('Repair cost outliers count:', outliers.shape[0])
outliers[['defect_id','repair_cost','severity','defect_type']].head()

In [ ]:
# Quick cross-tab: defect_type vs severity
pd.crosstab(df['defect_type'], df['severity'], normalize='index')

## Next steps
- Clean or impute any missing dates (if present).
- Consider log-transform of `repair_cost` for skewed distributions.
- Build simple predictive models (e.g., predict severity or cost) if needed.
- Export cleaned data to `defects_cleaned.csv`.